# CSHL Introductory Data Analysis Notebook

created by Adam S. Charles and Esther M. Whang for CSHL2025

This notebook will review how to utilize Python to analyze video data. Before proceeding, please ensure that you are using the CSHL2025 conda environment (please check the upper-right hand corner of your VScode and select the appropriate kernel)

## Loading and Viewing Data

The first step in data analysis is actually loading the data! Let's load up our data and see what we have.

In [ ]:
########################################################################################
# LOAD SOME DATA

## First make sure to load the necessary packages
from skimage import io  # skimage is a package for dealing with images. io = in/out
import numpy as np      # numpy is the core package for numerical processing

# This line will load a tif file from the Allen Institute.
# imread will check the tif header to look for file details (single vs. double etc.)

test_movie = io.imread('/home/aaa_shared/cshl2025/Data/Somatic/TEST_MOVIE_00001-small.tif')

# To check the data validity, look at basics of the data, i.e. the size of the data you loaded
print(test_movie.shape)         # Check the size of the data

# This data is in the shape of [frames]x[image height]x[image width]
########################################################################################

In [ ]:
########################################################################################
# Now let's look at the data 

# Import matplotlib: python's attempt to steal MATLAB's beauty
from matplotlib import pyplot as plt  

# Show the first frame of the image (remember that python counts from 0, vs. MATLAB that 
# counts from 1). 


frame_number = 0 
plt.imshow(test_movie[frame_number,:,:])        
color_bar = plt.colorbar()
color_bar.set_label('Intensity')
# EXPLORATORY POINT: try changing the frame number value around to look at different frames of the video!

########################################################################################

In [ ]:
########################################################################################
# The data is a little hard to see. There's some background etc. 
# Let's try to change the image contrast.
# First we need a sense of what the data range is: let's plot a histogram

number_of_bins = 2000 # determines the number of bins in the histogram
cnts = plt.hist(test_movie.ravel(),bins=number_of_bins)

# labels
plt.title("Histogram of Intensity Values for Video")
plt.ylabel("Count")
plt.xlabel("Intensity Value")

# EXPLORATORY POINT: Try to change the number of bins to see how the histogram looks. What do you notice as
# the number of bins exceeds 2000?

########################################################################################

In [ ]:
########################################################################################
# Now let's look at the data again but using the mode as the minimum and a low number
# as the max value.

min_value = 255 # mode of the distribution of the video
max_value = 900 # some low number
plt.imshow(test_movie[0,:,:],vmin=min_value , vmax=max_value)  
color_bar = plt.colorbar()
color_bar.set_label('Intensity')

# EXPLORATORY POINT: #Try seeing how the data looks when you change the mininum and maximum value!


########################################################################################

In [ ]:
########################################################################################
# There's lots of ways to look at the data. Try out some of the following

# fano, short for "fano factor", the standard deviation over the mean of some time frame.
selectSummary = "fano"   # Try switching between "mean", "median", "max" and "fano"

match selectSummary:
    case "mean":
        summaryImage = test_movie.mean(axis=0) 
    case "max":
        summaryImage = test_movie.max(axis=0)
    case "median":
        summaryImage = np.median(test_movie,axis=0)
    case "fano":
        summaryImage = 20*np.var(test_movie,axis=0)/(test_movie.mean(axis=0)*2) #the 20 is to normalize


plt.imshow(summaryImage, vmin=255, vmax=900)
color_bar = plt.colorbar()
color_bar.set_label('Intensity')

# EXPLORATORY POINT: Compare the fano image with the other options by changing selectSummary variable
# When might we want the "mean" image? the "fano" image? "max" image?

## Looking at Time Traces
We've identified some interesting neurons from exploring our data. Let's now see how we can use Python to look at their activity over time.

In [ ]:
########################################################################################
# We can now start zooming in on specific neurons.

# We've chosen a neuron for you that looks promising: once you've gone through this notebook once, 
# feel free to find your own neuron!

plt.imshow(summaryImage[45:70,345:365]) # an example neuron from the image above (35 by 20 pixel patch)
# EXPLORATORY POINT: Can you locate this patch in the summary image above? Why might it make a good example?

In [ ]:
########################################################################################
# Let's take a look at the time course of this region to figure out if it looks like a cell:

# Remember that our patch is from the rows 45:70 and columns 345:365 of each frame.

# One way of looking at the time course of the neuron is look at a pixel within this cropped image patch
single_pixel_trace = test_movie[:,53,355]
tt1 = plt.plot(single_pixel_trace, color='k', label='Single Pixel Time Trace')      # A single pixel's time-trace

# Another way to look at the time course is to take the average of the pixel values within the patch in each frame
local_avg_trace = np.average(test_movie[:,45:70,345:365], axis=[1,2]) # takes the average along the 2nd and 3rd dimension
tt2 = plt.plot(local_avg_trace, color='b', label='Averaged Time Trace')      # An average time-trace
plt.legend()

plt.xlabel("Time (Frames)")
plt.ylabel("Intensity")
########################################################################################

In [ ]:
########################################################################################
# The previous images are all looking at per-pixel statistics. What about cross-pixel?

corrImage = np.zeros((test_movie.shape[1],test_movie.shape[2]))   # Initialize the image to zeros

# let's look at the same pixel as above
pixleSelect = np.array([53,355])                  # Select which pixel to correlate to

for kk in range(test_movie.shape[1]):                     # Loop over rows
    for ll in range(test_movie.shape[2]):                 # Loop over columns
        tmp = np.corrcoef(test_movie[:,kk,ll],test_movie[:,pixleSelect[0],pixleSelect[1]])
        corrImage[kk,ll] = tmp[0,1]
        

plt.imshow(corrImage)    # Plot the correlation image
color_bar = plt.colorbar()
color_bar.set_label('Correlation Value')

# Now we see what other parts of the image correlates with the time traces of this one pixel!

## Alternative Plotting Libraries

Matplotlib, which we've been calling as plt, is a useful and popular plotting library. However, matplotlib is not the ONLY way to plot. Other tools exist that may be better suited for your particular purposes. We will be exploring some alternatives in this section -- if you are interested in adding additional libraries into the CSHL2025 conda environment, let us know!

In [ ]:
########################################################################################
# Let's try the plotly package
import plotly.express as px


fig = px.imshow(corrImage)  
fig.show()

# EXPLORATORY PONT: Try to mouse over time image. What's different from the above images?


In [ ]:
########################################################################################
# Summary images are useful, but often the full video is more useful

# plotly lets you look at the entire video
fig = px.imshow(test_movie[0:500,:,:], zmin=255, zmax=900, animation_frame=0, binary_string=True, labels=dict(animation_frame="slice"))
fig.show()

########################################################################################

In [ ]:
########################################################################################
# Sometimes data comes in a mat file:
# Could be, e.g., that you 

# Just for reference: if your .mat file is older, you can import it through scipy
# import scipy.io
# matFileData = scipy.io.loadmat('/home/aaa_shared/cshl2025/Data/Somatic/SEUDO_J122_2015-11-16_L01_c01_b27.mat')


# for more recent .mat files (matlab v7.3 files), you will need h5py
# first, let's check what's in the .mat file

import h5py
with h5py.File('/home/aaa_shared/cshl2025/Data/Somatic/SEUDO_J122_2015-11-16_L01_c01_b27.mat', 'r') as f:
    print(list(f.keys()))   # you should see ['#refs#', '#subsystem#', 'cellFile', 'dFF']

    # from the keys, let's take a look at the contents of 'cellFile'
    dataset = f['dFF']


########################################################################################

In [ ]:
########################################################################################
# Exercises

# Find a way to circle a cell that looks "good" in the 500x500 field of view data. 
# Average the pixels to create a time-series for that cell
# Repeat for 4 other cells and compute the correlation matrix between all time-traces

# For the cells you circled above, average the pixels AROUND the ROI you drew.
# Compute the correlations between the now 10 time traces

# For the traces above look at the histograms of all cell traces. What do you notice? 



########################################################################################